# [실습 4] Trainer API 파인튜닝 흐름 구성


## 실습 개요

> "학습 루프를 직접 작성하지 않고도 파인튜닝 파이프라인을 구성할 수 있을까?"

이번 실습에서는 `Trainer`, `TrainingArguments`, `DataCollatorWithPadding`, `compute_metrics`를 연결합니다.  
실제 긴 학습을 수행하기보다 Trainer가 요구하는 데이터셋 형식과 설정 객체의 역할을 이해하는 데 초점을 둡니다.


## 실습 목표

1. 텍스트 분류용 Dataset을 만든다.
2. 토크나이징된 컬럼과 labels 컬럼을 준비한다.
3. `TrainingArguments` 주요 옵션을 설정한다.
4. `compute_metrics` 콜백 함수를 구성한다.
5. [TODO] Trainer 객체를 완성한다.


## 데이터 출처

- 데이터: 실습용 문장/라벨 직접 구성
- 모델: `hf-internal-testing/tiny-random-bert`


## 목차

1. 라이브러리 임포트
2. Dataset 준비
3. 토크나이징과 data collator
4. TrainingArguments와 compute_metrics
5. [TODO] Trainer 구성


---
## 1. 라이브러리 임포트


### Trainer에 필요한 구성 요소 준비

Trainer를 사용하려면 모델, 토크나이저, 데이터셋, 배치 구성기, 학습 설정, 평가 함수가 서로 맞물려야 합니다. 먼저 공통으로 사용할 라이브러리와 작은 테스트 모델을 준비합니다.

여기서 사용하는 모델은 학습 품질을 확인하기 위한 모델이 아니라 Trainer 연결 구조를 빠르게 이해하기 위한 모델입니다. 실무 모델로 바꾸더라도 각 구성 요소가 맡는 역할은 동일합니다.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

checkpoint = 'hf-internal-testing/tiny-random-bert'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
print('준비 완료')

Loading weights: 100%|██████████| 89/89 [00:00<00:00, 10331.08it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: hf-internal-testing/tiny-random-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
qa_outputs.bias                            | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
qa_outputs.weight                          | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

준비 완료


---
## 2. Dataset 준비


### 학습과 검증용 Dataset 만들기

텍스트와 라벨을 가진 작은 Dataset을 만들고 train/validation split으로 나눕니다. Trainer는 Dataset에서 샘플을 읽어 배치로 묶고 모델에 전달하므로, 컬럼 구조가 중요합니다.

텍스트 분류에서는 정답 컬럼 이름을 보통 `labels`로 둡니다. Transformers 모델은 `labels`가 들어오면 내부에서 손실을 계산할 수 있어 Trainer 학습 루프와 자연스럽게 연결됩니다.

실무에서는 이 단계에서 라벨 분포, 중복 샘플, train과 validation의 누수 여부도 함께 확인합니다. 데이터셋이 작을수록 한두 개 샘플의 오류가 평가 결과에 크게 영향을 줄 수 있습니다.

In [2]:
rows = [
    {'text': 'The pipeline is easy to use.', 'labels': 1},
    {'text': 'The configuration is confusing.', 'labels': 0},
    {'text': 'Trainer reduces boilerplate code.', 'labels': 1},
    {'text': 'The preprocessing failed.', 'labels': 0},
    {'text': 'Metrics help compare experiments.', 'labels': 1},
    {'text': 'The batch is too large.', 'labels': 0},
]

dataset = Dataset.from_list(rows)
dataset_dict = DatasetDict({
    'train': dataset.select([0, 1, 2, 3]),
    'validation': dataset.select([4, 5]),
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 4
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 2
    })
})

---
## 3. 토크나이징과 data collator


### 토크나이징과 동적 패딩 연결하기

원문 텍스트는 토크나이징을 거쳐 모델 입력 컬럼으로 바뀝니다. `DataCollatorWithPadding`은 배치를 만들 때 그 배치에서 가장 긴 샘플에 맞춰 padding을 적용합니다.

동적 패딩은 모든 샘플을 항상 최대 길이로 맞추는 방식보다 불필요한 padding을 줄여 줍니다. 긴 문장과 짧은 문장이 섞인 데이터셋에서 학습 속도와 메모리 효율을 개선하는 데 도움이 됩니다.

`remove_columns=['text']`로 원문 텍스트 컬럼을 제거하는 이유도 확인해 볼 만합니다. 모델이 받지 않는 문자열 컬럼이 남아 있으면 배치 구성이나 forward 호출 과정에서 불필요한 오류가 생길 수 있습니다.

In [3]:
def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True)

tokenized = dataset_dict.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized['train'].column_names)
print(tokenized['train'][0])

Map: 100%|██████████| 2/2 [00:00<00:00, 201.63 examples/s]

['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']
{'text': 'The pipeline is easy to use.', 'labels': 1, 'input_ids': [2, 62, 811, 790, 58, 800, 805, 790, 806, 800, 794, 790, 51, 803, 47, 796, 803, 810, 62, 795, 63, 803, 790, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


---
## 4. TrainingArguments와 compute_metrics


### 학습 설정과 평가 콜백 설계

`TrainingArguments`는 학습률, 배치 크기, epoch 수, 저장 전략, 로깅 방식처럼 학습 루프의 동작을 정의합니다. 실험 재현성과 비용에 직접 영향을 주기 때문에 실무에서는 이 설정을 명확히 기록해 둡니다.

작은 실습에서는 epoch 수와 배치 크기를 작게 잡아 구조를 빠르게 확인합니다. 실제 파인튜닝에서는 데이터 규모, GPU 메모리, 검증 주기, 체크포인트 저장 정책을 함께 고려해 값을 조정합니다.

`compute_metrics`는 평가 단계에서 logits와 labels를 받아 원하는 지표를 반환하는 콜백입니다. Trainer가 평가 루프를 실행할 때 이 함수를 호출하므로, accuracy뿐 아니라 f1, precision, recall 같은 지표도 함께 계산하도록 확장할 수 있습니다.

In [ ]:
%pip install transformers[torch]
%pip install 'accelerate>=1.1.0'

In [12]:
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir='./tmp/hf_trainer_practice',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy='epoch',        # 또는 'steps'
    report_to='none',
)

print(training_args.output_dir)

./tmp/hf_trainer_practice


---
### [TODO] Trainer 객체 구성하기

아래 조건에 맞게 `trainer` 객체를 완성하세요.

구현 조건:
- `model`, `training_args`를 연결합니다.
- `train_dataset`은 `tokenized['train']`입니다.
- `eval_dataset`은 `tokenized['validation']`입니다.
- `tokenizer`, `data_collator`, `compute_metrics`를 함께 전달합니다.


### Trainer 객체로 학습 파이프라인 묶기

앞에서 준비한 모델, 학습 설정, 데이터셋, 토크나이저, collator, 평가 함수를 하나의 Trainer로 연결합니다. Trainer는 학습 반복문, 평가 루프, 배치 처리, 손실 계산, 체크포인트 저장 같은 공통 작업을 대신 관리합니다.

중요한 점은 Trainer가 데이터를 자동으로 알아서 해석하는 도구는 아니라는 것입니다. 모델이 기대하는 입력 이름과 Dataset 컬럼 구조가 맞아야 하고, collator가 만드는 배치도 모델의 forward 인자와 맞아야 합니다.

이 단계에서 `train_dataset`, `eval_dataset`, `data_collator`, `compute_metrics`가 같은 입력 형식을 기준으로 연결되는지 확인해야 합니다. 연결이 잘못되면 학습 시작 후에야 오류가 나기 때문에, Trainer 객체를 완성하기 전후로 데이터 샘플과 배치 형태를 함께 점검하는 편이 좋습니다.

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Step,Training Loss
1,0.695383
2,0.693260


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 30.71it/s]


TrainOutput(global_step=2, training_loss=0.6943213939666748, metrics={'train_runtime': 0.3429, 'train_samples_per_second': 11.666, 'train_steps_per_second': 5.833, 'total_flos': 26137464.0, 'train_loss': 0.6943213939666748, 'epoch': 1.0})

### Trainer 연결 상태 확인하기

학습을 시작하기 전에 Trainer 객체가 제대로 만들어졌는지, 학습/검증 데이터 크기가 의도대로 들어갔는지 확인합니다. `trainer.train()`을 바로 호출하기보다 이런 기본 연결 상태를 먼저 보는 편이 디버깅에 유리합니다.

객체 타입과 데이터셋 크기만 확인해도 TODO에서 Trainer 인자가 빠졌는지, split이 잘못 연결되었는지 같은 기본 문제를 빠르게 찾을 수 있습니다.

In [9]:
print(type(trainer).__name__)
print('train size:', len(trainer.train_dataset))
print('eval size:', len(trainer.eval_dataset))

Trainer
train size: 4
eval size: 2
